# US 4.6 -- Failure Mode Analysis

Aggregate metrics (F1, IoU) tell us *how well* the model performs, but not
*where* it fails.  This notebook shows how to:

1. Categorise every prediction as TP, FP, FN, or TN at the image level
2. Inspect the worst failure cases visually
3. Generate a structured failure report

## CLI equivalent

```bash
udm-epic4 report --checkpoint outputs/epic4_dann/best.pth \
                  --config configs/epic4_evaluate.yaml \
                  --n-visual 20
```

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from udm_epic4.models.dann import DANNModel
from udm_epic4.data.multi_domain_dataset import DomainDataset
from udm_epic4.reporting.failure_analysis import categorize_failures, generate_failure_report

---
## 1. Image-Level Failure Categories

Each image is classified based on whether the model and ground truth agree
on **void presence**:

| Category | Model predicts void? | Ground truth has void? | Meaning |
|----------|---------------------|----------------------|--------|
| **TP** | Yes | Yes | Correct detection |
| **FP** | Yes | No | False alarm |
| **FN** | No | Yes | Missed defect |
| **TN** | No | No | Correct rejection |

An image counts as "positive" if more than 0.01% of pixels are foreground.

FN (missed defects) are the most critical failures in quality inspection --
they mean real defects are not caught.

---
## 2. Using `categorize_failures`

The function runs inference on an entire dataset and returns a DataFrame
with one row per image.

In [ ]:
# Full workflow with real data:
#
# model = load_dann_model('outputs/epic4_dann/best.pth')
# dataset = DomainDataset('data/malaysia_eval/images', 'data/malaysia_eval/masks',
#                         domain_label=1, image_size=(512, 512))
#
# failures_df = categorize_failures(
#     model,
#     dataset,
#     device='cuda',
#     threshold=0.5,    # binarisation threshold
#     batch_size=8,
# )
#
# The returned DataFrame has columns:
# - image_path: path to the source image
# - category:   'TP', 'FP', 'FN', or 'TN'
# - pred_area:  fraction of predicted foreground pixels
# - true_area:  fraction of ground truth foreground pixels
# - iou:        pixel-level IoU for this image
# - notes:      human-readable description for failure cases

print("categorize_failures() returns a DataFrame with columns:")
print("  image_path, category, pred_area, true_area, iou, notes")

In [ ]:
# Simulated failure categorisation results
np.random.seed(42)
n_samples = 100

categories = np.random.choice(['TP', 'FP', 'FN', 'TN'], size=n_samples,
                               p=[0.35, 0.10, 0.15, 0.40])
pred_areas = np.random.uniform(0, 0.1, n_samples)
true_areas = np.random.uniform(0, 0.1, n_samples)
ious = np.random.uniform(0, 1, n_samples)

# Make IoU consistent with categories
for i in range(n_samples):
    if categories[i] == 'TN':
        ious[i] = 1.0
        pred_areas[i] = 0.0
        true_areas[i] = 0.0
    elif categories[i] == 'FP':
        ious[i] = 0.0
        true_areas[i] = 0.0
    elif categories[i] == 'FN':
        ious[i] = 0.0
        pred_areas[i] = 0.0

sim_df = pd.DataFrame({
    'image_path': [f'data/images/img_{i:04d}.png' for i in range(n_samples)],
    'category': categories,
    'pred_area': pred_areas,
    'true_area': true_areas,
    'iou': ious,
})

# Summary
print("Failure categorisation summary:")
print(sim_df['category'].value_counts().to_string())
print(f"\nTotal samples: {len(sim_df)}")

---
## 3. Visual Examples of TP / FP / FN

The `generate_failure_report` function saves PNG overlays of the worst
predictions.  Each visual shows:
- **Left:** input image
- **Centre:** ground truth mask
- **Right:** model prediction with category label and IoU

In [ ]:
# Simulated visual examples (with synthetic data)
fig, axes = plt.subplots(3, 3, figsize=(12, 12))

examples = [
    ('TP -- Correct Detection', 0.82),
    ('FP -- False Alarm', 0.00),
    ('FN -- Missed Defect', 0.00),
]

np.random.seed(123)
for row, (title, iou_val) in enumerate(examples):
    # Simulate image, ground truth, prediction
    img = np.random.rand(64, 64) * 0.5 + 0.25
    gt = np.zeros((64, 64))
    pred = np.zeros((64, 64))

    if 'TP' in title:
        gt[20:40, 20:40] = 1.0
        pred[22:42, 18:38] = 1.0  # slightly shifted
    elif 'FP' in title:
        pred[10:25, 10:25] = 1.0  # false positive
    elif 'FN' in title:
        gt[30:50, 30:50] = 1.0     # missed

    axes[row, 0].imshow(img, cmap='gray')
    axes[row, 0].set_title('Input', fontsize=10)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(gt, cmap='gray', vmin=0, vmax=1)
    axes[row, 1].set_title('Ground Truth', fontsize=10)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(pred, cmap='hot', vmin=0, vmax=1)
    axes[row, 2].set_title(f'{title}\nIoU={iou_val:.2f}', fontsize=10)
    axes[row, 2].axis('off')

fig.suptitle('Failure Categories -- Visual Examples', fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

---
## 4. Generating the Full Report

The `generate_failure_report` function writes three artefacts to disk:

| Output | Description |
|--------|------------|
| `failures.csv` | Full categorised DataFrame |
| `summary.txt` | Aggregate statistics (counts, precision, recall, IoU) |
| `visuals/` | PNG overlays of the N worst failures (by IoU) |

In [ ]:
# Full report generation workflow:
#
# failures_df = categorize_failures(model, dataset, device='cuda')
#
# generate_failure_report(
#     failures_df,
#     output_dir='outputs/epic4_evaluation/failure_report/malaysia',
#     model=model,       # needed for visual overlays
#     dataset=dataset,   # needed for visual overlays
#     device='cuda',
#     n_visual=20,       # save 20 worst-case visuals
# )

print("Report outputs:")
print("  failures.csv   -- full per-image categorisation")
print("  summary.txt    -- aggregate statistics")
print("  visuals/       -- PNG overlays of worst failures")

In [ ]:
# Show what the summary.txt looks like
summary_text = """
Failure Analysis Summary
========================================
Total samples analysed: 100

Category counts:
  TP (true positive):  35
  FP (false positive): 10
  FN (false negative): 15
  TN (true negative):  40

Image-level precision: 0.7778
Image-level recall:    0.7000

Mean IoU (all samples):      0.6820
Mean IoU (positive samples): 0.4530

TP statistics (n=35):
  Mean pred area:  0.0412
  Mean true area:  0.0380
  Mean IoU:        0.7620

FP statistics (n=10):
  Mean pred area:  0.0185
  Mean true area:  0.0000
  Mean IoU:        0.0000

FN statistics (n=15):
  Mean pred area:  0.0000
  Mean true area:  0.0290
  Mean IoU:        0.0000
"""
print(summary_text)

---
## 5. Interpreting the Results

### Key questions to ask:

1. **Are FN cases concentrated in a specific domain?** If so, DANN may not
   have fully adapted to that domain.

2. **What do FP cases look like?** Common causes:
   - Artefacts in the image (scratches, dust) confused for voids.
   - Over-sensitive model after DANN training.

3. **What do FN cases look like?** Common causes:
   - Very small voids below the detection threshold.
   - Voids in unusual locations or with unusual appearance.
   - Domain-specific void patterns not captured by the source training data.

4. **What is the IoU distribution for TP cases?** Low TP IoU means the model
   detects voids but with poor spatial precision.

### Actions based on findings:

| Finding | Action |
|---------|--------|
| Many FP | Increase binarisation threshold; add negative examples |
| Many FN | Lower threshold; augment training data; train longer |
| Low TP IoU | Increase decoder capacity; use higher resolution |
| Domain-specific failures | Add more target domain data to DANN training |

---
## 6. Per-Domain Failure Reports

The CLI generates a separate report for each evaluation domain:

```bash
udm-epic4 report --checkpoint outputs/epic4_dann/best.pth \
                  --config configs/epic4_evaluate.yaml
```

Output structure:
```
outputs/epic4_evaluation/failure_report/
  warstein/
    failures.csv
    summary.txt
    visuals/
  malaysia/
    failures.csv
    summary.txt
    visuals/
  regensburg/
    ...
```

Compare the summaries across domains to identify which sites need the
most attention.

---
## Summary

- `categorize_failures()` classifies every image as TP/FP/FN/TN.
- `generate_failure_report()` writes CSV, summary stats, and visual overlays.
- Focus on FN cases (missed defects) -- these are the most critical in production.
- Use per-domain reports to identify which sites need more adaptation.

This is the final notebook in the Epic 4 series.  For the full pipeline:

| Step | Notebook | CLI |
|------|---------|-----|
| 1. Understand DANN | `epic4_00_overview` | -- |
| 2. Analyse data | `epic4_01_data_analysis` | `udm-epic4 prepare` |
| 3. Train baseline | `epic4_02_baseline` | `udm-epic4 baseline` |
| 4. Train DANN | `epic4_03_dann_training` | `udm-epic4 train` |
| 5. Analyse features | `epic4_04_domain_analysis` | `udm-epic4 analyze` |
| 6. Evaluate | `epic4_05_deployment` | `udm-epic4 evaluate` |
| 7. Failure report | `epic4_06_failure_analysis` | `udm-epic4 report` |